In [1]:
import os
import sys
from argparse import ArgumentParser, BooleanOptionalAction
import warnings
import json
import torch
import logging
import random
import numpy as np
import re
import time
import pandas as pd
import datetime as dt
from tqdm import tqdm
import logging.config
from datasets import Dataset
# from transformers.utils import logging
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, EarlyStoppingCallback

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/home/shaib.c/.conda/envs/mds_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/shaib.c/.conda/envs/mds_env/lib/python3.10/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


/home/shaib.c/.conda/envs/mds_env/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cpu.so: undefined symbol: cadam32bit_grad_fp32


RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'is_mlu_available' from 'accelerate.utils' (/home/shaib.c/.conda/envs/mds_env/lib/python3.10/site-packages/accelerate/utils/__init__.py)

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)

cache_path = "/scratch/wadhwa.s/pattern_distillation/"

In [ ]:
m = "/scratch/wadhwa.s/pattern_distillation/pattern_distill_models/trained/cnn/gpt2_mistral7binstruct/checkpoint-1000"
tokenizer = AutoTokenizer.from_pretrained(m)
model = AutoModelForCausalLM.from_pretrained(m, 
                                            cache_dir=cache_path,
                                            # trust_remote_code = remote_code,
                                            local_files_only = True,
                                            device_map="auto")

In [ ]:
df = df = pd.read_csv("/work/frink/shaib.c/pattern_distillation/inference/cnn/gpt2_mistral7b.csv")
df.head()

In [ ]:
df["clm"] = tokenizer.bos_token + df["text"] + " #### [SUMMARY]"

In [ ]:
d_test = Dataset.from_pandas(df.sample(5))

In [ ]:
d_test

In [ ]:
processed = []
gold = []
ids = []
article = []
teacher = []
for ins in tqdm(d_test):
    m_input = ins["clm"]
    inputs = tokenizer(m_input, return_tensors="pt").input_ids.to(device)
    outputs = model.generate(inputs, 
                            max_length=1024, 
                            # do_sample=True, 
                            # top_k=50, 
                            # top_p=0.95, 
                            # temperature=0.7,
                            num_return_sequences=1,
                            use_cache=True,
                            pad_token_id=tokenizer.pad_token_id,
                            # eos_token_id=tokenizer.eos_token_id,
                            # bos_token_id=tokenizer.bos_token_id,
                            # no_repeat_ngram_size=2,
                            # early_stopping=True,
                            # num_beams=5,
                            # length_penalty=1.0,
                            )
    torch.cuda.empty_cache()
    generated_ids = outputs.to('cpu')
    generated_tokens = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)
    processed.append(generated_tokens)
    gold.append(ins["gold_summary"])
    teacher.append(ins["generated_summary"])
    article.append(ins["text"])
    ids.append(ins["id"])
    

In [ ]:
for summ, gold, teacher, article, ids in zip(processed, gold, teacher, article, ids):
    op = summ[0]
    match = re.search(r'\[SUMMARY\]\s*(.*?)\s*\[SUMMARY\]', op.strip())
    if match:
        summary = match.group(1)
    else:
        summary = "No summary found"
    print("Student Summary: ", summary)
    print ("Teacher Summary: ", teacher)
    print("Gold Summary: ", gold)
    print("ID: ", ids)
    print("\n---\n")

In [ ]:
df = pd.read_csv("/work/frink/shaib.c/pattern_distillation/inference/cnn/gpt2_llama70b.csv")
df.head()

In [ ]:
for ix, row in df.iterrows():
    print ("ID: ", row["id"])
    print ("\nStudent Summary: ", row["student"])
    print ("\nTeacher Summary: ", row["teacher_summ"])
    print ("\n-------------------\n")

In [3]:
x = pd.read_csv('/work/frink/shaib.c/pattern_distillation/original_data/pubmed_summ.csv')

In [4]:
x.token_length.mean()

634.5106

In [5]:
len(x)

5000

In [6]:
x

,article,abstract,token_length
0,a 59-year - old asymptomatic diabetic male was...,we report a case of unusually long persistence...,463
1,\n the preoperative diagnosis was intussuscep...,objective : laparoscopic reduction of appendic...,763
2,sodium valproate ( sv ) has a simple chemical ...,sodium valproate ( sv ) is effective and well ...,773
3,the prisma-7 tool has been introduced in two...,introductionthe prisma-7 tool [ 1 ] has been i...,293
4,"\n fourteen children , 7 dyslexics and 7 cont...",objectives : to investigate the characteristic...,192
...,...,...,...
4995,it is now widely accepted that patients in the...,antithrombotic prophylaxis in critically ill p...,951
4996,"\n i thank momoko ohori , kenichi iwai , yusu...",abstractthe molecular mechanism responsible fo...,102
4997,sirenomelia is a rare and fatal congenital ano...,sirenomelia also known as the mermaid syndrome...,830
4998,progressive symmetric erythrokeratoderma ( pse...,progressive symmetric erythrokeratoderma ( pse...,898


In [7]:
5000 * 634 

3170000

In [9]:
3170000 /1000000

3.17

In [10]:
3.17 * 0.05

0.1585